## YOLO-Pose 26m Training

### Loading data and installation of Ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!unzip "/content/drive/MyDrive/dataset_3.zip" -d /content/

In [ ]:
!pip install ultralytics

In [ ]:
!find /content/dataset_3 -type f | head

In [ ]:
from pathlib import Path

img_dir = Path("dataset_3/images/train")
lbl_dir = Path("dataset_3/labels/train")

imgs = sorted([x.stem for x in img_dir.glob("*")])
lbls = sorted([x.stem for x in lbl_dir.glob("*.txt")])

print("Images:", len(imgs))
print("Labels:", len(lbls))

missing = [x for x in imgs if x not in lbls]

print("Missing labels:", len(missing))
print(missing[:10])

In [ ]:
p = next(lbl_dir.glob("*.txt"))
print(p.read_text())

### Training Loop

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26m-pose.pt")

model.train(
    data="/content/dataset_3/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16
)

### Prediction and labelling of unseen data

In [ ]:
!unzip "/content/unlabelled_4.zip" -d /content/

In [ ]:
from ultralytics import YOLO

model = YOLO("best_m.pt")

model.predict(
    source="unlabelled_4",
    save=True,
    save_txt=True,
    conf=0.25
)

In [ ]:
import shutil
from pathlib import Path

PRED_LABELS_DIR = Path("runs/pose/predict/labels")  # change to your predicted labels path
IMAGES_DIR      = Path("unlabelled_4")             # your images folder

output_images = Path("predicted_package/images")
output_labels = Path("predicted_package/labels")
output_images.mkdir(parents=True, exist_ok=True)
output_labels.mkdir(parents=True, exist_ok=True)

saved = 0
missing = 0

for label_file in PRED_LABELS_DIR.glob("*.txt"):
    stem = label_file.stem
    image_file = None
    for ext in [".png", ".jpg", ".jpeg", ".PNG", ".JPG"]:
        candidate = IMAGES_DIR / (stem + ext)
        if candidate.exists():
            image_file = candidate
            break

    if image_file is None:
        print(f"MISSING image: {stem}")
        missing += 1
        continue

    shutil.copy2(image_file, output_images / image_file.name)
    shutil.copy2(label_file, output_labels / label_file.name)
    saved += 1

shutil.make_archive("predicted_package", "zip", "predicted_package")

print(f"Saved  : {saved}")
print(f"Missing: {missing}")

from google.colab import files
files.download("predicted_package.zip")